# 03 — Model Training (MIL 3D CNN)

This notebook handles the training of a from-scratch 3D CNN using Multiple Instance Learning (MIL) on the UCF-Crime dataset. It includes support for AMP training, MIL ranking loss, and resume-safe checkpointing for local environment.

In [ ]:
import os, sys, json

BASE_DIR = ".."\n
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from training.config import ProjectConfig, set_seed, get_device, save_config
from training.model.scratch_3dcnn import Scratch3DCNN, count_parameters
from training.loss import MILRankingLoss
from training.dataset.transforms import VideoTransform
from training.dataset.mil_dataset import (
    UCFCrimeManifest, MILBagDataset, MILBagValidationDataset,
    mil_collate_fn, val_collate_fn
)
from training.train import train, load_checkpoint
from training.metrics import compute_video_level_auc

print('All imports successful!')

In [ ]:
config = ProjectConfig()

# Override any settings for this run
config.SEED = 42  # Change to 123, 456 for multi-seed runs
config.EPOCHS = 60
config.PATIENCE = 12
config.BATCH_SIZE = 4
config.AMP = True

set_seed(config.SEED)
device = get_device()
print(f'Device: {device}')
print(f'Config: seed={config.SEED}, lr={config.LR}, epochs={config.EPOCHS}')

# Save run config
run_dir = os.path.join(config.RUNS_DIR, f'run_seed{config.SEED}')
os.makedirs(run_dir, exist_ok=True)
save_config(config, os.path.join(run_dir, 'run_config.json'))

## 1. Model Architecture

Here we initialize the from-scratch 3D CNN model.

In [ ]:
model = Scratch3DCNN().to(device)
count_parameters(model)

# Quick shape smoke test
with torch.no_grad():
    dummy = torch.randn(2, 3, 16, 112, 112).to(device)
    out = model(dummy)
    print(f'\nSmoke test - Input: {dummy.shape} -> Output: {out.shape}')
    
    # Test with inference spatial size
    dummy_infer = torch.randn(2, 3, 16, 112, 144).to(device)
    out_infer = model(dummy_infer)
    print(f'Inference test - Input: {dummy_infer.shape} -> Output: {out_infer.shape}')

print('\nModel architecture:')
print(model)

## 2. Loss Function

In [ ]:
criterion = MILRankingLoss(
    margin=config.RANKING_MARGIN,
    lambda_sparse=config.LAMBDA_SPARSE,
    lambda_smooth=config.LAMBDA_SMOOTH,
    topk=config.TOPK,  # None = max-MIL, set to 3 for top-k MIL
)
print(f'Loss: margin={config.RANKING_MARGIN}, topk={config.TOPK}')
print(f'Regularization: lambda_sparse={config.LAMBDA_SPARSE}, lambda_smooth={config.LAMBDA_SMOOTH}')

# Smoke test loss on logits
with torch.no_grad():
    anom_logits = torch.randn(4, 32)
    norm_logits = torch.randn(4, 32)
    loss_out = criterion(anom_logits, norm_logits)
    print(f'\nLoss smoke test: {loss_out}')

## 3. Data Loading

In [ ]:
# Load manifests
train_manifest = UCFCrimeManifest(os.path.join(BASE_DIR, 'manifest_train.csv'))
val_manifest = UCFCrimeManifest(os.path.join(BASE_DIR, 'manifest_val.csv'))

print(f'Train: {len(train_manifest.normal_videos)} normal, {len(train_manifest.anomalous_videos)} anomalous')
print(f'Val: {len(val_manifest.normal_videos)} normal, {len(val_manifest.anomalous_videos)} anomalous')

# Transforms
train_transform = VideoTransform(mode='train')
val_transform = VideoTransform(mode='val')

# Datasets
train_dataset = MILBagDataset(
    manifest=train_manifest,
    transform=train_transform,
    clips_per_bag=config.CLIPS_PER_BAG,
    clip_length=config.CLIP_LENGTH,
    target_fps=config.TARGET_FPS,
)

val_dataset = MILBagValidationDataset(
    manifest=val_manifest,
    transform=val_transform,
    clip_length=config.CLIP_LENGTH,
    clip_stride=config.CLIP_STRIDE,
    target_fps=config.TARGET_FPS,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=mil_collate_fn,
    pin_memory=True,
    drop_last=True,
)

print(f'\nTrain batches per epoch: {len(train_loader)}')
print(f'Validation videos: {len(val_dataset)}')

## 4. Training Loop

In [ ]:
# Check for existing checkpoint to resume
last_ckpt = os.path.join(config.CHECKPOINT_DIR, f'last_model_seed{config.SEED}.pt')
best_ckpt = os.path.join(config.CHECKPOINT_DIR, f'best_model_seed{config.SEED}.pt')

if os.path.exists(last_ckpt):
    print(f'Found existing checkpoint: {last_ckpt}')
    print('Training will resume from this checkpoint.')
else:
    print('No existing checkpoint found. Starting from scratch.')

In [ ]:
best_model_path = train(
    config=config,
    model=model,
    train_loader=train_loader,
    val_dataset=val_dataset,
    criterion=criterion,
    device=device,
)
print(f'\nBest model saved to: {best_model_path}')

## 5. Training Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_path = os.path.join(config.RUNS_DIR, f'run_seed{config.SEED}', 'training_log.csv')
if os.path.exists(log_path):
    log_df = pd.read_csv(log_path)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Total loss
    axes[0,0].plot(log_df['epoch'], log_df['train_total'], label='Train')
    if 'val_total' in log_df.columns:
        axes[0,0].plot(log_df['epoch'], log_df['val_total'], label='Val')
    axes[0,0].set_title('Total Loss')
    axes[0,0].legend()
    axes[0,0].set_xlabel('Epoch')
    
    # Ranking loss
    axes[0,1].plot(log_df['epoch'], log_df['train_rank'], label='Rank Loss')
    axes[0,1].set_title('Ranking Loss')
    axes[0,1].set_xlabel('Epoch')
    
    # Validation AUC
    if 'val_auc' in log_df.columns:
        axes[1,0].plot(log_df['epoch'], log_df['val_auc'], 'g-', label='Val AUC')
        axes[1,0].set_title('Validation AUC')
        axes[1,0].legend()
        axes[1,0].set_xlabel('Epoch')
    
    # Learning rate
    if 'lr' in log_df.columns:
        axes[1,1].plot(log_df['epoch'], log_df['lr'])
        axes[1,1].set_title('Learning Rate')
        axes[1,1].set_xlabel('Epoch')
    
    plt.suptitle(f'Training Curves (Seed {config.SEED})', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, 'training_curves.png'), dpi=150)
    plt.show()
    print(f'Best val AUC: {log_df["val_auc"].max():.4f} at epoch {log_df["val_auc"].idxmax()}')
else:
    print(f'Training log not found at {log_path}')

Deliverable summary.